# Clase 1 · Ingesta y capa Bronze

Vamos a generar cuatro fuentes, inspeccionar sus formatos y materializarlas como tablas Delta Bronze. Bronze conserva el dato recibido: los problemas se identifican, pero se corrigen recién en Silver.

In [0]:
dbutils.widgets.text("student_id", "alumno01", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
data = build_course_data(spark, config)
paths = write_landing_files(data, config)
paths

{'customers_csv': '/Volumes/workspace/bigdata_drrach/landing/customers_csv',
 'products_parquet': '/Volumes/workspace/bigdata_drrach/landing/products_parquet',
 'transactions_csv': '/Volumes/workspace/bigdata_drrach/landing/transactions_csv',
 'events_json': '/Volumes/workspace/bigdata_drrach/landing/events_json'}

## 1. Leer no es todavía transformar

Inspeccioná los cuatro orígenes. Compará el esquema inferido del CSV con el esquema físico de Parquet. ¿Por qué `amount` termina como texto?

In [0]:
customers_raw = spark.read.option("header", True).csv(paths["customers_csv"])
transactions_raw = spark.read.option("header", True).csv(paths["transactions_csv"])
products_raw = spark.read.parquet(paths["products_parquet"])
events_raw = spark.read.json(paths["events_json"])

for name, frame in {"customers": customers_raw, "transactions": transactions_raw, "products": products_raw, "events": events_raw}.items():
    print(f"\n--- {name} ---")
    frame.printSchema()


--- customers ---
root
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- email: string (nullable = true)


--- transactions ---
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- is_fraud: string (nullable = true)


--- products ---
root
 |-- product_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- price: decimal(12,2) (nullable = true)


--- events ---
root
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullab

### Experimento: inferencia de tipos

Volvé a leer transacciones con `inferSchema=true`. Registrá cuánto tarda y verificá si resuelve correctamente `amount`. Explicá por qué inferir implica trabajo adicional y puede producir decisiones inestables.

In [0]:
transactions_inferred = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(paths["transactions_csv"]))
transactions_inferred.printSchema()
display(transactions_inferred.limit(10))

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- amount: string (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- is_fraud: integer (nullable = true)



transaction_id,customer_id,product_id,event_ts,amount,payment_channel,device_id,is_fraud
500,10,28,2026-03-06T11:24:57.000Z,1267.30,transfer,device_41,0
501,63,12,2026-03-04T23:39:06.000Z,1742.30,transfer,device_44,0
502,2,1,2026-03-05T04:34:11.000Z,674.49,transfer,device_9,0
503,13,6,2026-03-03T00:15:41.000Z,718.37,wallet,device_4,0
504,34,5,2026-03-02T08:42:37.000Z,2000.67,transfer,device_4,1
505,6,20,2026-03-03T10:05:29.000Z,659.26,card,device_46,0
506,91,27,2026-03-01T13:20:43.000Z,114.28,card,device_31,0
507,99,24,2026-03-06T09:53:44.000Z,416.02,transfer,device_1,0
508,99,20,2026-03-05T09:51:42.000Z,1191.30,transfer,device_1,0
509,49,29,2026-03-03T09:36:07.000Z,637.62,transfer,device_48,0


## 2. Metadatos de ingesta

Bronze agrega trazabilidad sin modificar la semántica del dato de origen.

In [0]:
from pyspark.sql import functions as F

def with_ingestion_metadata(frame, source_name, source_path):
    return (frame
        .withColumn("_source", F.lit(source_name))
        .withColumn("_source_file", F.lit(source_path))
        .withColumn("_ingested_at", F.current_timestamp()))

bronze = {
    "customers": with_ingestion_metadata(customers_raw, "customers_csv", paths["customers_csv"]),
    "products": with_ingestion_metadata(products_raw, "products_parquet", paths["products_parquet"]),
    "transactions": with_ingestion_metadata(transactions_raw, "transactions_csv", paths["transactions_csv"]),
    "events": with_ingestion_metadata(events_raw, "events_json", paths["events_json"]),
}

In [0]:
for name, frame in bronze.items():
    table = f"{config.catalog}.{config.schema}.bronze_{name}"
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(table)
    print(f"{table}: {spark.table(table).count():,} filas")

workspace.bigdata_drrach.bronze_customers: 100 filas
workspace.bigdata_drrach.bronze_products: 30 filas
workspace.bigdata_drrach.bronze_transactions: 1,001 filas
workspace.bigdata_drrach.bronze_events: 3,000 filas


## 3. Diagnóstico inicial de calidad

No limpies todavía. Medí cuántos identificadores están duplicados y cuántos importes no pueden convertirse a número.

In [0]:
tx = spark.table(f"{config.catalog}.{config.schema}.bronze_transactions")
quality_summary = tx.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("distinct_ids"),
    F.sum(F.expr("CASE WHEN try_cast(amount AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END")).alias("invalid_amounts"),
)
display(quality_summary)

rows,distinct_ids,invalid_amounts
1001,1000,3


In [0]:
quality_summary.show(truncate=False)

+----+------------+---------------+
|rows|distinct_ids|invalid_amounts|
+----+------------+---------------+
|1001|1000        |3              |
+----+------------+---------------+



## 4. Delta y el plan de ejecución

Ejecutá `DESCRIBE DETAIL`, `DESCRIBE HISTORY` y `EXPLAIN FORMATTED` sobre una tabla. Identificá qué información agrega Delta respecto de un directorio Parquet.

In [0]:
table_name = f"{config.catalog}.{config.schema}.bronze_transactions"
display(spark.sql(f"DESCRIBE DETAIL {table_name}"))
display(spark.sql(f"DESCRIBE HISTORY {table_name}"))
spark.sql(f"EXPLAIN FORMATTED SELECT payment_channel, count(*) FROM {table_name} GROUP BY payment_channel").show(truncate=False)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,e0d2ba52-692d-4cf7-bb93-72322e2fb882,workspace.bigdata_drrach.bronze_transactions,null,,2026-09-26T15:20:29.374Z,2026-09-28T02:27:24.000Z,List(),List(),1,16688,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> b8b848f7-c56f-47e4-82f7-909d4ccb37a1)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-28T02:27:24.000Z,79130279751925,racheluja@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""b8b848f7-c56f-47e4-82f7-909d4ccb37a1""}, statsOnLoad -> true)",null,List(2293625909945710),ce9c94be-d39f-425d-99b7-d7dff9b1bca0,0928-022535-3nld7mse-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 687066, numDeletionVectorsRemoved -> 0, numOutputRows -> 1001, numOutputBytes -> 16688)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-26T15:34:09.000Z,79130279751925,racheluja@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""b8b848f7-c56f-47e4-82f7-909d4ccb37a1""}, statsOnLoad -> true)",null,List(2293625909945710),5089f843-2fb2-4056-90cf-803a0c1d7fa5,0926-150815-blcdnzth-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 687066, numDeletionVectorsRemoved -> 0, numOutputRows -> 50011, numOutputBytes -> 687066)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-26T15:20:32.000Z,79130279751925,racheluja@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""b8b848f7-c56f-47e4-82f7-909d4ccb37a1""}, statsOnLoad -> true)",null,List(2293625909945710),5fb5db09-72eb-41f5-88ae-863c9f652434,0926-150815-blcdnzth-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50011, numOutputBytes -> 687066)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Entrega breve

Dejá ejecutadas en este notebook las salidas de las cuatro tablas y del diagnóstico de calidad. Las respuestas escritas van en el `README.md` de tu entrega: (1) tres observaciones sobre CSV/JSON, Parquet y Delta; y (2) una explicación de dónde aparece volumen, velocidad, variedad, veracidad y valor en este caso. El formato completo está en la sección **Formato de entrega** del README de la práctica.